In [ ]:
!pip install pandas plotly openpyxl


In [ ]:
import pandas as pd
import numpy as np
import os

def clean_data(df):


    df = df.drop_duplicates()


    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # Convert numeric columns safely
    if "quantity" in df.columns:
        df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

    if "unit_price" in df.columns:
        df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    # Convert date
    if "order_date" in df.columns:
        df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", dayfirst=True)

    # Validate required columns
    required_cols = ["quantity", "unit_price", "order_date", "order_id"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {', '.join(missing_cols)}")


    # Remove invalid rows
    df = df.dropna(subset=["quantity", "unit_price", "order_date"])
    df = df[df["quantity"] > 0]
    df = df[df["unit_price"] > 0]

    return df


In [ ]:

import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import streamlit as st

def get_summary_statistics(df):
    """Returns basic descriptive statistics for the dataframe."""
    return df.describe()

def visualize_numerical_distributions(df):
    """Plots histograms and box plots for numerical columns."""
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
    charts = {}

    for col in numerical_cols:
        if col in ['order_id']: continue # Skip ID columns

        # Histogram
        fig_hist = px.histogram(df, x=col, title=f"Distribution of {col}", template="plotly_dark", color_discrete_sequence=['#00CC96'])

        # Box Plot
        fig_box = px.box(df, y=col, title=f"Box Plot of {col}", template="plotly_dark", color_discrete_sequence=['#EF553B'])

        charts[col] = {"histogram": fig_hist, "boxplot": fig_box}

    return charts

def visualize_categorical_counts(df):
    """Plots bar charts for categorical columns."""
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    charts = {}

    for col in categorical_cols:
        if df[col].nunique() > 20: continue # Skip high cardinality columns for clarity

        # Bar Chart
        counts = df[col].value_counts().reset_index()
        counts.columns = [col, 'count']
        fig = px.bar(counts, x=col, y='count', title=f"Count of {col}", template="plotly_dark", color='count', color_continuous_scale='Viridis')

        charts[col] = fig

    return charts

def visualize_time_series(df, date_col):
    """Plots time series trends if a date column exists."""
    if date_col not in df.columns:
        return None

    # Resample by day or month depending on range
    df = df.sort_values(by=date_col)

    # Check if we have numeric columns to plot over time
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    # Remove potential ID columns
    if 'order_id' in numeric_cols: numeric_cols.remove('order_id')

    if not numeric_cols: return None

    charts = {}
    for col in numeric_cols:
        daily_agg = df.groupby(date_col)[col].sum().reset_index()
        fig = px.line(daily_agg, x=date_col, y=col, title=f"{col} Over Time", template="plotly_dark", markers=True)
        charts[col] = fig

    return charts

def visualize_correlation_heatmap(df):
    """Plots a correlation heatmap for numerical columns."""
    numeric_df = df.select_dtypes(include=['float64', 'int64'])

    # Remove ID columns if present
    if 'order_id' in numeric_df.columns:
        numeric_df = numeric_df.drop(columns=['order_id'])

    if numeric_df.shape[1] < 2:
        return None

    corr = numeric_df.corr()

    fig = px.imshow(
        corr,
        text_auto=True,
        aspect="auto",
        title="Correlation Heatmap",
        template="plotly_dark",
        color_continuous_scale='RdBu_r'
    )
    return fig



In [ ]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 91.2 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.1
    Uninstalling cachetools-7.0.1:
      Successfully uninstalled cachetools-7.0.1


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving cleaned_data (1).csv to cleaned_data (1).csv


In [ ]:
import pandas as pd

file_name = list(uploaded.keys())[0]

if file_name.endswith(".csv"):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

df.head()


,order_id,order_date,sku,color,size,unit_price,quantity,revenue
0,1.0,01-06-2022 16:05,708,Dark Blue,2XL,298.0,1.0,298.0
1,1.0,01-06-2022 16:05,89,Dark Blue,2XL,258.0,1.0,258.0
2,2.0,10-06-2022 17:03,bobo,Cream,One Size,228.0,1.0,228.0
3,3.0,10-06-2022 19:59,799,Dark Blue,XL,264.0,1.0,264.0
4,4.0,11-06-2022 11:42,799,Dark Blue,M,264.0,1.0,264.0


In [ ]:
cleaned_df = clean_data(df)
cleaned_df.head()


,order_id,order_date,sku,color,size,unit_price,quantity,revenue
0,1.0,2022-06-01 16:05:00,708,Dark Blue,2XL,298.0,1.0,298.0
1,1.0,2022-06-01 16:05:00,89,Dark Blue,2XL,258.0,1.0,258.0
2,2.0,2022-06-10 17:03:00,bobo,Cream,One Size,228.0,1.0,228.0
3,3.0,2022-06-10 19:59:00,799,Dark Blue,XL,264.0,1.0,264.0
4,4.0,2022-06-11 11:42:00,799,Dark Blue,M,264.0,1.0,264.0


In [ ]:
charts = visualize_numerical_distributions(cleaned_df)

for col in charts:
    charts[col]["histogram"].show()
    charts[col]["boxplot"].show()
